# Building a Data Lake: The `spatiotemporal_lake` Blueprint

In the BMC architecture, downloading, extracting, and standardizing raw environmental data is handled by an "Offline Ingestion Orchestrator" known as a Data Lake. 

To ensure that every new data source (whether it's Copernicus WEkEO, Google Earth Engine, or a local FTP server) behaves predictably, we use an Abstract Base Class (ABC) called `spatiotemporal_lake`. 

This base class acts as a **strict contract**. It inherits all the mathematical capabilities of the `spatial_engine`, but it forces any developer building a new data provider to implement four specific phases:

1. **Fetch:** Connect to the remote server and download raw files.
2. **Build:** Translate those raw files into standardized Cloud Optimized GeoTIFFs (COGs).
3. **Validate:** Perform strict quality assurance (QA) on the generated data.
4. **Catalog:** Create an inventory file for downstream runtime cubes to query.

### The Four Pillars of Ingestion

If you want to build a new data connector, your class *must* implement these four abstract methods. If you miss even one, Python will throw an error and refuse to instantiate your class.

* `fetch_raw_data(self, recipe, logger)`: This is where you put your API authentication and downloading logic. 
* `build_datalake(self, recipe, logger)`: This phase orchestrates the engine tools to mosaic raw tiles and calculate fractional coverages.
* `validate_datalake(self, base_dir, logger)`: This is your ecological gatekeeper. It must perform strict QA tests, ensuring that fractions sum to 100% and mathematical bounds are respected.
* `generate_catalog(self, target_dir, logger)`: Finally, this method crawls your finished directories to build a lightweight CSV/Parquet inventory.

You don't have to write the spatial math from scratch—the base class provides utilities like `gather_tifs_from_zips` and `process_virtual_mosaic` for you to use. You just have to wire them together.